In [1]:
# do all the nessesary setup, get the proper libraries installed
!pip install https://github.com/cterry311/TakRoadBuilder/raw/refs/heads/master/rust_game/wheels/tak_engine-1.1.0-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl


In [ ]:
# import nessesary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List
import numpy as np
import tak_engine
import sqlite3
from google.colab import drive
import os
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

# connect to google drive

drive.mount('/content/drive/')

print(os.listdir('/content/drive/MyDrive'))
with open('/content/drive/MyDrive/capstone/test.txt', 'w') as file:
    file.write('hello world!')

with open('/content/drive/MyDrive/capstone/test.txt', 'r') as file:
    test_content = file.read()

if test_content != 'hello world!':
    raise AssertionError('something went wrong with write')
print('setup done')


Mounted at /content/drive/
['old stuff', 'capstone', 'War of the Stray Dog   Incident at Petrich.gslides', 'email.gdoc', 'Curriculum.gdoc']
setup done


In [ ]:
# set up the classes and functions imported from other files



# TakNet


# CNN, has 6x6 input
# it has the following channels
# for the top layer, it has 6 channels, indicating flats, caps, and walls for each player, each on has a bianary representation, either 0 or 1
# there are 6 layers below that represented, each layer represented with 2 channels, one for white, and one for black.
# 1 layer encoding the stack height at that point, compressed within the range [0, 1)
# 1 layer for flat count differential
# 1 layer for each player representing if they have played their capstone yet
# 1 layer for each player representing the preportion of stones they have played.
# an all 1s plane for clearly identifying the board edge

# 251 policy output channels,
# 3 of those channels are place moves
# the rest are directional



IN_CHANNELS = 6 + (2 * 6) + 1 + 1 + 2 + 2 + 1

TRUNK_CHANNELS = 64
TRUNK_BLOCKS = 6

POLICY_OUTPUT_CHANNELS = 251
POLICY_BLOCKS = 1

VALUE_NET_SIZE = 64
VALUE_BLOCKS = 1



class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(ch)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)


class TakNet(nn.Module):
    def __init__(self, in_ch=IN_CHANNELS, tr_ch=TRUNK_CHANNELS, tr_bl=TRUNK_BLOCKS, pol_ch=POLICY_OUTPUT_CHANNELS, pol_bl=POLICY_BLOCKS, val_size=VALUE_NET_SIZE, val_bl=VALUE_BLOCKS,):
        super().__init__()
        trunk_ops: List[nn.Module] = [
            nn.Conv2d(in_ch, tr_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(tr_ch),
            nn.ReLU(),
        ]
        for _ in range(tr_bl):
            trunk_ops.append(ResBlock(tr_ch))
        self.trunk = nn.Sequential(*trunk_ops)

        policy_ops: List[nn.Module] = []
        for _ in range(pol_bl):
            policy_ops.append(ResBlock(tr_ch))
        policy_ops.append(nn.Conv2d(tr_ch, pol_ch, 1))
        self.policy = nn.Sequential(*policy_ops)

        value_ops: List[nn.Module] = [
            nn.Conv2d(tr_ch, 3, 1, bias=False),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * 6 * 6, val_size),
            nn.ReLU(),
        ]
        for _ in range(val_bl):
            value_ops.append(nn.Linear(val_size, val_size))
            value_ops.append(nn.ReLU())
        value_ops.append(nn.Linear(val_size, 3))
        # value_ops.append(nn.Softmax(dim=1)) #value head is now logits instead of probabilities
        self.value = nn.Sequential(*value_ops)

    def forward(self, x):
        x = self.trunk(x)
        return self.policy(x), self.value(x) # value output, 0 = loss, 1 = draw, 2 = win









# Must match the dtype used when writing. Defined here (not imported) because
# importing your self-play file would run its module-level DB setup.
VISIT_DTYPE = np.dtype([("move", "<u2"), ("n", "<u4")])
POLICY_LEN = 251 * 36


def split_game_ids(db_path, val_frac=0.05, seed=42):
    """Split by game, never by row. Capped games (outcome 0) are excluded."""
    conn = sqlite3.connect(db_path)
    ids = [r[0] for r in conn.execute("SELECT id FROM self_play_games WHERE outcome != 0")]
    conn.close()
    rng = np.random.default_rng(seed)
    rng.shuffle(ids)
    n_val = max(1, int(len(ids) * val_frac))
    return ids[n_val:], ids[:n_val]          # train_ids, val_ids


class SelfPlayDataset(Dataset):
    def __init__(self, db_path, game_ids=None):
        conn = sqlite3.connect(db_path)
        conn.row_factory = sqlite3.Row

        games = {g["id"]: dict(g) for g in
                 conn.execute("SELECT * FROM self_play_games WHERE outcome != 0")}
        if game_ids is not None:
            keep = set(game_ids)
            games = {k: v for k, v in games.items() if k in keep}
        if not games:
            raise ValueError("no usable games in database")
        self.games = games
        self.metadata = self._build_metadata()

        rows = conn.execute(
            "SELECT game_id, ply, tps, move_raw, sampled, q, visits FROM self_play_positions"
        ).fetchall()
        conn.close()
        self.rows = [tuple(r) for r in rows if r[0] in games]

    def _build_metadata(self):
        g = list(self.games.values())
        komis = {x["komi"] for x in g}
        versions = {x["encoder_version"] for x in g}
        if len(komis) != 1 or len(versions) != 1:
            raise ValueError(f"mixed settings: komi={komis}, encoder_version={versions}")
        return {
            "komi": komis.pop(),
            "encoder_version": versions.pop(),
            "models": sorted({x["model"] for x in g}),
            "max_sims": sorted({x["max_sims"] for x in g}),
            "n_games": len(g),
            "n_positions_expected": sum(x["n_plies"] for x in g),
        }

    @property
    def komi(self):
        return self.metadata["komi"]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        game_id, ply, tps, move_raw, sampled, q, blob = self.rows[i]
        outcome = self.games[game_id]["outcome"]       # 1 white, 2 black, 3 draw

        # policy target: normalized visits, dense over the 9036 layout
        v = np.frombuffer(blob, dtype=VISIT_DTYPE)
        idx = tak_engine.bulk_raw_to_index(v["move"].tolist())
        target = np.zeros(POLICY_LEN, dtype=np.float32)
        target[idx] = v["n"]
        target /= target.sum()

        # value class from the mover's view: 0 loss, 1 draw, 2 win
        mover = 1 if (ply & 1) == 0 else 2             # white moves on even plies
        if outcome == 3:
            value_class = 1
        else:
            value_class = 2 if outcome == mover else 0

        return (
            tps,                                       # str (collates to a list)
            torch.from_numpy(target),                  # (9036,) float32
            tak_engine.raw_to_index(move_raw),         # index of the move actually played
            value_class,                               # 0/1/2, matches the value head
            float(q),                                  # root q, mover's view
            int(sampled),
        )


In [ ]:
# method for running self play

def setup_db():
    conn = sqlite3.connect("/content/self_play_temp.db")
    conn.execute("PRAGMA journal_mode=WAL")
    conn.executescript("""
    DROP INDEX IF EXISTS idx_pos_game;
    DROP TABLE IF EXISTS self_play_games;
    DROP TABLE IF EXISTS self_play_positions;

    CREATE TABLE IF NOT EXISTS self_play_games (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        seed INTEGER NOT NULL,            -- masked to < 2^63
        model TEXT NOT NULL,
        encoder_version INTEGER NOT NULL,
        komi INTEGER NOT NULL,
        max_sims INTEGER NOT NULL,
        batch_size INTEGER NOT NULL,
        c_puct REAL NOT NULL,
        urgency REAL NOT NULL,
        alpha REAL NOT NULL,
        eps REAL NOT NULL,
        temp_plies INTEGER NOT NULL,      -- how many of the first n plys get a randomly sampled move
        n_plies INTEGER NOT NULL, -- how many moves were made in the game
        outcome INTEGER NOT NULL,         -- 0 ongoing 1 white, 2 black, 3 draw
        end_reason TEXT
    );
    CREATE TABLE IF NOT EXISTS self_play_positions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        game_id INTEGER NOT NULL REFERENCES self_play_games(id),
        ply INTEGER NOT NULL,
        tps TEXT NOT NULL,
        move_raw INTEGER NOT NULL,        -- the move actually played
        sampled INTEGER NOT NULL,         -- 1 if sampled with temperature, 0 if argmax
        q REAL NOT NULL,                  -- root q, mover's view
        visits BLOB NOT NULL
    );
    CREATE INDEX IF NOT EXISTS idx_pos_game ON self_play_positions(game_id);
    """)
    conn.commit()
    return conn

ENCODER_VERSION = 2


VISIT_DTYPE = np.dtype([("move", "<u2"), ("n", "<u4")])   # 2 + 4 = 6 bytes per entry

def pack_visits(visit_counts):
    """visit_counts: list of (move_raw, visits). Keeps only visited moves."""
    arr = np.array([(m, n) for m, n in visit_counts if n > 0], dtype=VISIT_DTYPE)
    return arr.tobytes()

def unpack_visits(blob):
    arr = np.frombuffer(blob, dtype=VISIT_DTYPE)
    return [(int(m), int(n)) for m, n in arr]          # back to a list of tuples


def sample_move(visit_counts, seed: int, T=0.2):
    rng = np.random.default_rng(seed)
    moves = np.array([m for m, n in visit_counts])
    visits = np.array([n for m, n in visit_counts], dtype=np.float64)
    p = visits ** (1.0 / T)
    p /= p.sum()
    return int(rng.choice(moves, p=p))

class Player:
    def __init__(self, model, komi, max_sims, batch_size, c_puct, urgency, alpha, eps, temp_plies):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        info = torch.load(model, map_location=self.device)
        self.model = TakNet(**info["params"])
        self.model.load_state_dict(info["weights"])
        self.model.to(self.device).eval()  # eval() matters: BatchNorm must not use batch stats

        self.BATCH = batch_size
        self.buf = np.zeros((self.BATCH, 25, 6, 6), dtype=np.float32)
        self.flat = self.buf.reshape(-1)  # a view onto buf; Rust writes through it

        self.komi = komi
        self.max_sims = max_sims
        self.batch_size = batch_size
        self.c_puct = c_puct
        self.urgency = urgency
        self.alpha = alpha
        self.eps = eps
        self.temp_plies = temp_plies

    @torch.no_grad()
    def gen_move(self, tps: str, ply: int, seed: int) -> tuple[int, list[tuple[int, int]], float]:
        # generate the move, return the following (move_raw, visit_counts, q), each entry in visit count is as follows (move_raw, visits)
        s = tak_engine.Search(tps, urgency=self.urgency, c_puct=self.c_puct, komi=self.komi)
        if s.is_over():
            return None
        i = 0
        while True:
            if s.sims() >= self.max_sims:
                break

            n = s.collect(self.flat, self.BATCH)
            if n == 0:  # every descent hit a finished game; already backed up
                continue

            x = torch.from_numpy(self.buf[:n]).to(self.device)
            policy, value = self.model(x)

            pol = torch.softmax(policy.flatten(1), dim=1).cpu().numpy().reshape(-1)
            vp = torch.softmax(value, dim=1)
            val = (vp[:, 2] - vp[:, 0]).cpu().numpy()

            s.submit(pol, val)  # row i here must be row i of buf
            if i == 0:
                s.add_noise(self.alpha, self.eps, seed)
            i += 1
        visit_counts = [(x[0], x[1]) for x in s.root_stats()]
        if ply < self.temp_plies:
            return sample_move(visit_counts, seed), visit_counts, s.root_q()
        else:
            best = max(visit_counts, key=lambda x: x[1])[0]
            return best, visit_counts, s.root_q()


def copy_db(conn: sqlite3.Connection, copy_path):
    other = sqlite3.connect(copy_path)
    conn.backup(other)
    other.close()


def play_self(model, komi, max_sims, batch_size, c_puct, urgency, alpha, eps, temp_plies, max_ply, num_games, conn, save_path):
    print('starting...')
    player = Player(model, komi, max_sims, batch_size, c_puct, urgency, alpha, eps, temp_plies)
    print('model loaded...')
    rand = np.random.default_rng()
    for j in range(num_games):
        game = tak_engine.Tak(komi=komi)
        seed = int(rand.integers(0, 1 << 63))
        game_records = []
        i = 0
        end_reason = ''
        while True:
            tps = game.to_tps()
            mv, visit_counts, q = player.gen_move(tps, i, seed + i)
            game_records.append((tps, i, mv, visit_counts, q))
            game.make_move_raw(mv)
            i += 1
            if game.result() != 0:
                end_reason = 'game_over'
                break
            if i >= max_ply:
                end_reason = 'max_ply'
                break
        game_id = conn.execute('''
            INSERT INTO self_play_games (
                seed,   
                model,
                encoder_version,
                komi,
                max_sims,
                batch_size,
                c_puct,
                urgency,
                alpha,
                eps,
                temp_plies,    
                n_plies, 
                outcome,   
                end_reason
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            seed,
            model,
            ENCODER_VERSION,
            komi,
            max_sims,
            batch_size,
            c_puct,
            urgency,
            alpha,
            eps,
            temp_plies,
            i,
            game.result(),
            end_reason
        )).lastrowid
        for record in game_records:
            # (tps, i, mv, visit_counts, q)
            conn.execute('''
                INSERT INTO self_play_positions (
                    game_id,
                    ply,
                    tps,
                    move_raw,      
                    sampled,    
                    q,        
                    visits
                )
                VALUES (?, ?, ?, ?, ?, ?, ?)
            ''', (
                game_id,
                record[1],
                record[0],
                record[2],
                1 if record[1] < temp_plies else 0,
                record[4],
                pack_visits(record[3])
            ))
        conn.commit()
        print(f'Game {game_id} written to database.')
        if j % 10 == 0:
            copy_db(conn, f'/content/drive/MyDrive/capstone/records/{save_path}')

def run_self_play(model_path, save_path):
    model_path = f"/content/drive/MyDrive/capstone/models/{model_path}"
    conn = setup_db()
    play_self(model_path, 2, 5_000, 256, 1.5, -0.2, 0.05, 0.25, 14, 300, 150, conn, save_path)
    print(f'finished self play with model {model_path}')
    conn.commit()
    conn.close()


# run_self_play('/content/drive/MyDrive/capstone/version_9_5.pt')


# if __name__ == "__main__":
#     play_self('model_checkpoints/version_9_1.pt', 2, 5_000, 256, 1.5, -0.2, 0.05, 0.25, 20, 300, 1000)

['.config', 'drive', 'self_play_temp.db', 'sample_data']
['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']
['old stuff', 'capstone', 'War of the Stray Dog   Incident at Petrich.gslides', 'email.gdoc', 'Curriculum.gdoc']
['games.db', 'version_9_5.pt', 'anothertest.txt', 'test.txt', 'test_data.db']
starting...
model loaded...
Game 3 written to database.
Game 4 written to database.
Game 5 written to database.
Game 6 written to database.
Game 7 written to database.
Game 8 written to database.
Game 9 written to database.
Game 10 written to database.
Game 11 written to database.
Game 12 written to database.
Game 13 written to database.
Game 14 written to database.
Game 15 written to database.
Game 16 written to database.
Game 17 written to database.
Game 18 written to database.
Game 19 written to database.
Game 20 written to database.
Game 21 written to database.
Game 22 written to database.
Game 23 written to database.
Game 24 written to database.
Game 25 written to database.

KeyboardInterrupt: 

In [ ]:
# method for doing the renforcment learning


# ---------------- settings ----------------


EPOCHS = 4
BATCH = 128
LR = 3e-4                             # lower than supervised (guess, tune)
W_V = 0.5                             # value cross-entropy on game outcome
GAMMA = 0.25                          # value vs search q
BETA = 0.1                            # advantage term; set 0 to disable
ADV_CLIP = 1.0
VAL_FRAC = 0.05
ENCODER_VERSION = 2                   # must match what the data was generated with

# ---------------- augmentation (copied from supervised file) ----------------
DIRS = [(1, 0), (-1, 0), (0, -1), (0, 1)]          # engine order + - < >
TRANSFORMS = [(k, flip) for flip in (False, True) for k in range(4)]

def apply_T(t, k, flip):
    t = torch.rot90(t, k, dims=(-2, -1))
    return torch.flip(t, dims=(-1,)) if flip else t

def build_perms():
    base = torch.arange(36).reshape(6, 6)
    perms = []
    for k, flip in TRANSFORMS:
        moved = apply_T(base, k, flip).flatten()
        new_of_old = torch.empty(36, dtype=torch.long)
        new_of_old[moved] = torch.arange(36)
        a = divmod(int(new_of_old[2 * 6 + 2]), 6)
        dir_map = []
        for dy, dx in DIRS:
            b = divmod(int(new_of_old[(2 + dy) * 6 + 2 + dx]), 6)
            dir_map.append(DIRS.index((b[0] - a[0], b[1] - a[1])))
        perm = torch.empty(251 * 36, dtype=torch.long)
        for ch in range(251):
            if ch < 3:
                nch = ch
            else:
                s = ch - 3
                nch = 3 + (s // 4) * 4 + dir_map[s % 4]
            for cell in range(36):
                perm[ch * 36 + cell] = nch * 36 + int(new_of_old[cell])
        perms.append(perm)
    return perms

def augment(x, legal, target, played, perms):
    """x (n,25,6,6); legal (n,9036) bool; target (n,9036) float; played (n,) int64. All on one device.
    Value targets are invariant, so they are not touched."""
    n = x.shape[0]
    ts = torch.randint(0, 8, (n,), device=x.device)
    x_o, legal_o = x.clone(), torch.empty_like(legal)
    target_o, played_o = torch.empty_like(target), played.clone()
    for t, (k, flip) in enumerate(TRANSFORMS):
        idx = (ts == t).nonzero(as_tuple=True)[0]
        if idx.numel() == 0:
            continue
        perm = perms[t]
        x_o[idx] = apply_T(x[idx], k, flip)
        legal_o[idx[:, None], perm[None, :]] = legal[idx]       # new[perm[i]] = old[i]
        target_o[idx[:, None], perm[None, :]] = target[idx]
        played_o[idx] = perm[played[idx]]
    return x_o, legal_o, target_o, played_o

# ---------------- loss ----------------
def compute_losses(policy, value_logits, legal, target, played, value_class, q, sampled):
    logits = policy.flatten(1).masked_fill(~legal, -1e9)
    logp = F.log_softmax(logits, dim=1)

    policy_loss = -(target * logp).sum(1).mean()
    value_ce = F.cross_entropy(value_logits, value_class)
    probs = F.softmax(value_logits, dim=1)
    value_q = F.mse_loss(probs[:, 2] - probs[:, 0], q)

    # advantage term: z - q, from the mover's view, no gradient through the advantage
    z = value_class.float() - 1.0                              # 0/1/2 -> -1/0/+1
    A = (z - q).clamp(-ADV_CLIP, ADV_CLIP).detach()
    logp_a = logp.gather(1, played[:, None]).squeeze(1)
    p_a = logp_a.exp().clamp(max=1 - 1e-6)
    pos = A.clamp(min=0) * (-logp_a)                           # won more than expected: raise move
    neg = (-A).clamp(min=0) * (-torch.log1p(-p_a))             # lost more than expected: lower move


    #adv = ((pos + neg) * sampled).mean()                       # sampled moves only
    adv = (pos + neg).mean()  # all moves have this loss applied, not just sampled


    total = policy_loss + W_V * value_ce + GAMMA * value_q + BETA * adv

    return total, dict(policy=policy_loss, value_ce=value_ce, value_q=value_q, adv=adv, tiltak_losses=tiltak_losses)

# ---------------- batch preparation ----------------
def prepare(batch, bufs, komi, device):
    tps, target, played, value_class, q, sampled = batch
    n = len(tps)
    inputs, inputs_np, masks, masks_np = bufs
    tak_engine.encode_training_batch(tps, inputs_np[:n], masks_np[:n], komi=komi)
    x = inputs[:n].to(device)
    legal = masks[:n].reshape(n, -1).to(device).bool()
    return (x, legal, target.to(device), played.to(device).long(),
            value_class.to(device).long(), q.to(device).float(), sampled.to(device).float())



# ---------------- main ----------------
def run_training(LOAD_PATH, SAVE_PATH):
    DB_PATH = '/content/'
    torch.manual_seed(0)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("training on", device)

    train_ids, val_ids = split_game_ids(DB_PATH, VAL_FRAC)
    train_ds = SelfPlayDataset(DB_PATH, train_ids)
    val_ds = SelfPlayDataset(DB_PATH, val_ids)
    print("train:", train_ds.metadata)
    print("val games:", val_ds.metadata["n_games"], "positions:", len(val_ds))
    komi = train_ds.komi
    assert val_ds.komi == komi
    assert train_ds.metadata["encoder_version"] == ENCODER_VERSION, "encoder version mismatch"

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False)

    info = torch.load(f"/content/drive/MyDrive/capstone/models/{LOAD_PATH}", map_location=device)
    params = info["params"]
    model = TakNet(**params)
    model.load_state_dict(info["weights"])
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    perms = [p.to(device) for p in build_perms()]
    inputs = torch.empty(BATCH, 25, 6, 6)
    masks = torch.empty(BATCH, 251, 6, 6, dtype=torch.uint8)
    bufs = (inputs, inputs.numpy(), masks, masks.numpy())

    best = float("inf")
    for epoch in range(EPOCHS):
        model.train()
        sums, count = {}, 0
        for batch in train_loader:
            x, legal, target, played, vclass, q, sampled = prepare(batch, bufs, komi, device)
            x, legal, target, played = augment(x, legal, target, played, perms)

            optimizer.zero_grad()
            policy, value = model(x)
            loss, parts = compute_losses(policy, value, legal, target, played, vclass, q, sampled)
            loss.backward()
            optimizer.step()

            parts["total"] = loss
            for k, v in parts.items():
                sums[k] = sums.get(k, 0.0) + v.item()
            count += 1
        print(f"epoch {epoch} train " + " ".join(f"{k} {v / count:.4f}" for k, v in sums.items()))

        model.eval()
        vs, vcount, pol_acc, val_acc = {}, 0, 0.0, 0.0
        with torch.no_grad():
            for batch in val_loader:
                x, legal, target, played, vclass, q, sampled = prepare(batch, bufs, komi, device)
                policy, value = model(x)
                loss, parts = compute_losses(policy, value, legal, target, played, vclass, q, sampled)
                parts["total"] = loss
                for k, v in parts.items():
                    vs[k] = vs.get(k, 0.0) + v.item()
                logits = policy.flatten(1).masked_fill(~legal, -1e9)
                pol_acc += (logits.argmax(1) == target.argmax(1)).float().mean().item()
                val_acc += (value.argmax(1) == vclass).float().mean().item()
                vcount += 1
        print(f"epoch {epoch} val   " + " ".join(f"{k} {v / vcount:.4f}" for k, v in vs.items())
                + f" policy_acc {pol_acc / vcount:.4f} value_acc {val_acc / vcount:.4f}")

        # select on the terms that are comparable across runs (not the adv term)
        score = (vs["policy"] + W_V * vs["value_ce"]) / vcount
        if score < best:
            best = score
            torch.save({"params": params, "weights": model.state_dict(),
                        "komi": komi, "encoder_version": ENCODER_VERSION},
                        f"/content/drive/MyDrive/capstone/models/{SAVE_PATH}")
            print("saved best")



# if __name__ == "__main__":
#     print('starting')
#     run()


In [ ]:
# iterate between self play and training, save checkpoints to drive periodically

model_version = 9
model_sub_version = 5
while True:
    model_path = f'version_{model_version}_{model_sub_version}.pt'
    model_save_path = f'version_{model_version}_{model_sub_version + 1}.pt'

    db_path = f'records_{model_version}_{model_sub_version}.pt'

    run_self_play(model_path, db_path)
    run_training(model_path, model_save_path)

